# كشكول التجارب: قراءة وتقطيع النصوص (Text Ingestion & Splitting)

أهلاً بيك! في الكشكول ده هنجرب نقرأ الملفات النصية العادية (`.txt`) ونشوف إزاي نقطعها لفقرات صغيرة تناسب الـ RAG.

الفكرة ببساطة:
1. بنقرأ الملف من القرص بترميز سليم (`UTF-8`).
2. بنحوله لكائن اسمه `Document` جواه النص وميتا-داتا (زي مسار الملف).
3. بنجرب نقطعه باستخدام `RecursiveCharacterTextSplitter` عشان نحافظ على معنى الجمل من غير ما نقطع كلمة في النص!


In [ ]:
# 1. نستورد الأدوات ونحدد مسار البيانات
from pathlib import Path

# نحدد مسار مجلد data تلقائياً
repo_root = Path.cwd()
while not (repo_root / "data").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
data_dir = repo_root / "data"

from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

txt_folder = data_dir / "txt"
print(f"مكان ملفات النصوص: {txt_folder}")
print(f"الملفات الموجودة: {[f.name for f in txt_folder.glob('*.txt')]}")


### 2. تعال نقرأ ملف نصي واحد
هنستخدم `TextLoader` ونقرأ ملف عن الذكاء الاصطناعي:


In [ ]:
# نقرأ أول ملف
file_path = txt_folder / "artificial_intelligence.txt"
loader = TextLoader(str(file_path), encoding="utf-8")
docs = loader.load()

print(f"عدد المستندات: {len(docs)}")
print(f"طول النص بالكامل: {len(docs[0].page_content)} حرف")
print("\nأول 150 حرف من الملف:\n" + docs[0].page_content[:150] + "...")


### 3. تقطيع النص لفقرات ذكية
دلوقتي النص طويل ومينفعش نبعته كله مرة واحدة للـ LLM.  
هنستخدم `RecursiveCharacterTextSplitter` ونقوله: عايزين كل قطعة حوالي 200 حرف، وخلي فيه تداخل (`overlap`) 30 حرف بين كل قطعتين عشان المعنى ميفلتش!


In [ ]:
# نجهز أداة التقطيع
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

# نقطع المستند
chunks = splitter.split_documents(docs)

print(f"النص اتقطع لـ {len(chunks)} قطعة!")
print("\nتعال نشوف أول قطعتين:")
for i, chunk in enumerate(chunks[:2], 1):
    print(f"--- القطعة رقم {i} ({len(chunk.page_content)} حرف) ---")
    print(chunk.page_content.strip())


---
### مساحتك للتجربة الحرة
جرب في الخلية دي تغير حجم الـ `chunk_size` خليه مثلاً `100` أو `400`، أو جرب تقرأ ملف تاني من المجلد وشوف عدد القطع هيطلع كام!


In [ ]:
# جرب هنا براحتك:
# my_file = txt_folder / "software_engineering.txt"
# my_docs = TextLoader(str(my_file), encoding="utf-8").load()
# جرب قطعه وشوف النتيجة
